# 05 · Generation + guardrails: when should the tutor refuse?
Inspects `ragtraining.guard.rules` and the results of `ragtraining.eval.guardrails` (LOGBOOK §7.4). Reads stored results: **$0**.
Defences, in order: **① input filter** (regex) → **② confidence gate** (best rerank score too low → "not covered", no Opus call)
→ **③ chunk sanitizer** (drops instruction-like lines in documents) → **④ system prompt** (sources are data, G1–G11)
→ **⑤ citation validator**.

In [ ]:
import sys, json, pickle; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
from ragtraining.eval.guardrails import threshold_table
from ragtraining.generate.tutor import REFUSE_BELOW
sc = pd.read_csv('../data/guard_scores.csv')
suite = pd.DataFrame(json.load(open('../data/guard_suite.json')))
sc.groupby('kind')[['claude', 'local']].median()

## 1. The confidence gate: two score distributions
Answerable questions should score high, not-covered ones low. The **threshold** sits in the gap. Where the two overlap,
you have to choose which error is worse.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for a, col, bins, cut in [(ax[0], 'claude', np.arange(-0.5, 11), [3, 6]), (ax[1], 'local', np.logspace(-3.2, 0, 30), [0.05])]:
    for kind, c in [('answerable', 'tab:green'), ('off_topic', 'tab:red'), ('unanswerable', 'tab:orange')]:
        a.hist(sc[sc.kind == kind][col], bins=bins, alpha=.6, color=c, label=kind)
    for t in cut: a.axvline(t, ls='--', color='k'); a.text(t, a.get_ylim()[1] * .9, f' {t}', fontsize=8)
    a.set(title=f'{col} reranker: best score per question', xlabel='top score'); a.legend(fontsize=8)
ax[1].set_xscale('log'); plt.tight_layout(); plt.show()
threshold_table(sc)

**Reading:** on synthetic questions a threshold of 6 looks perfect. But the real, vague question *"What is a key?"*
scored **4** → it would be wrongly refused (**P-37**). The gate only saves money: whatever slips through still reaches Opus,
which refuses off-topic questions itself. So a **false refusal is the worse error** → final threshold **Claude 3, local 0.05**.

## 2. The suite: 26 cases end to end

In [ ]:
suite.groupby('kind', sort=False).passed.agg(passed='sum', cases='count')

In [ ]:
suite[['kind', 'question', 'passed', 'why']].style.hide(axis='index')

## 3. Injection hidden inside a document
The payload is appended to the top real chunk. The obvious English one matches the regex and gets **removed**. The subtle
English and Italian ones get through to Opus, which **ignores** them because the system prompt says sources are data (G5).

In [ ]:
from ragtraining.guard.rules import sanitize_chunk
for p in suite[suite.kind == 'injection_doc'].payload:
    clean, n = sanitize_chunk('Feistel: L_i = R_(i-1) ...\n' + p)
    print(f'removed {n} line(s) | payload: {p[:70]}')
print('\nAnswer with the subtle payload (look for "24 December": absent):\n')
print(suite[suite.kind == 'injection_doc'].answer.iloc[1][:600])

## 4. Ambiguous questions: retrieval decides the breadth of the answer (P-38)
For *"What is a key?"* all 8 sources are crypto, so the tutor can't mention **primary keys** (Data Management).
That's a **retrieval** problem and the motivation for multi-query (notebook 07).

In [ ]:
for _, r in suite[suite.kind == 'ambiguous'].iterrows():
    print(f"{'PASS' if r.passed else 'FAIL'} {r.question:26s} {r.why}")
    print('     sources:', sorted({s.split(' · ')[0].split(' ', 1)[1] for s in r.sources}))